Nombre: Emilio Rico Hernández
Clase: 4
Challenge: Data Cleaning Challenge
Fecha: 2026-09-25

# Challenge 4 — Data Cleaning: clientes de un servicio con app

Uso el mismo dataset del Challenge 2 (`clientes_app.csv`): 1,840 registros de clientes con plan, ciudad, uso de la app, gasto, satisfacción y churn. El objetivo es auditarlo, limpiarlo y documentar cada decisión. Mi regla es no cambiar nada que no pueda justificar: si un dato es sospechoso pero no sé cuál es el valor correcto, lo marco con una bandera en lugar de borrarlo o inventarlo.

Nota: este dataset no trae variantes tipo "Mexico / MEXICO / México" ni fechas imposibles; en los pasos 4 y 5 lo reviso igual y dejo escrito qué encontré.

## Paso 1 — Diagnóstico

In [1]:
import pandas as pd

df = pd.read_csv("clientes_app.csv", encoding="utf-8-sig")
df_original = df.copy()      # copia para comparar antes vs después
print("Filas y columnas:", df.shape)
df.info()

Filas y columnas: (1840, 21)
<class 'pandas.DataFrame'>
RangeIndex: 1840 entries, 0 to 1839
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_cliente         1840 non-null   str    
 1   fecha              1840 non-null   str    
 2   edad               1840 non-null   int64  
 3   ciudad             1816 non-null   str    
 4   plan               1840 non-null   str    
 5   canal_adquisicion  1840 non-null   str    
 6   antiguedad_meses   1840 non-null   int64  
 7   sesiones_mes       1840 non-null   int64  
 8   minutos_app        1803 non-null   float64
 9   tickets_soporte    1840 non-null   int64  
 10  satisfaccion       1794 non-null   float64
 11  descuento_pct      1840 non-null   int64  
 12  ingreso_mensual    1777 non-null   float64
 13  gasto_mensual      1840 non-null   float64
 14  compras_mes        1840 non-null   int64  
 15  mora_dias          1840 non-null   int64  
 16  califi

In [2]:
print(df.isna().sum())
print("Filas duplicadas:", df.duplicated().sum())

id_cliente            0
fecha                 0
edad                  0
ciudad               24
plan                  0
canal_adquisicion     0
antiguedad_meses      0
sesiones_mes          0
minutos_app          37
tickets_soporte       0
satisfaccion         46
descuento_pct         0
ingreso_mensual      63
gasto_mensual         0
compras_mes           0
mora_dias             0
calificacion          0
review               27
sentimiento           0
churn                 0
gasto_proximo_mes     0
dtype: int64
Filas duplicadas: 40


In [3]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_cliente,1840,1800,C00970,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fecha,1840,580,2025-09-02,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
edad,1840.0,NaN,NaN,NaN,37.917935,10.464747,18.0,30.0,38.0,45.0,72.0
ciudad,1816,5,CDMX,644,NaN,NaN,NaN,NaN,NaN,NaN,NaN
plan,1840,3,Básico,786,NaN,NaN,NaN,NaN,NaN,NaN,NaN
canal_adquisicion,1840,5,Orgánico,528,NaN,NaN,NaN,NaN,NaN,NaN,NaN
antiguedad_meses,1840.0,NaN,NaN,NaN,21.35163,18.560945,1.0,7.0,16.0,31.0,72.0
sesiones_mes,1840.0,NaN,NaN,NaN,14.079891,6.633711,0.0,9.0,14.0,19.0,36.0
minutos_app,1803.0,NaN,NaN,NaN,342.204382,483.858498,0.0,188.5,303.0,424.75,8751.6
tickets_soporte,1840.0,NaN,NaN,NaN,0.907065,0.911264,0.0,0.0,1.0,1.0,4.0


| Problema | Columna | Acción propuesta |
|---|---|---|
| Faltante (24) | `ciudad` | imputar con "Desconocida" |
| Faltante (63) | `ingreso_mensual` | conservar como NaN |
| Faltante (37) | `minutos_app` | conservar como NaN |
| Faltante (46) | `satisfaccion` | conservar como NaN |
| Faltante (27) | `review` | conservar (no escribió reseña) |
| Duplicado (40 filas) | todas | eliminar si es real |
| Texto | `ciudad`, `plan`, `canal_adquisicion`, `sentimiento` | revisar y normalizar si hace falta |
| Tipo | `fecha` (viene como texto) | convertir a datetime |
| Regla de negocio | edad, gasto, sesiones, minutos, calificación | revisar en el Paso 6 |
| Outlier | `ingreso_mensual`, `minutos_app`, `gasto_mensual` | investigar; conservar si son plausibles |

Todavía no modifiqué nada. Lo más frecuente son los faltantes: 197 celdas vacías repartidas en cinco columnas.

## Paso 2 — Valores faltantes

In [4]:
faltantes = df.isna().sum()
print((faltantes[faltantes > 0] / len(df) * 100).round(2))

df["ciudad"] = df["ciudad"].fillna("Desconocida")

ciudad             1.30
minutos_app        2.01
satisfaccion       2.50
ingreso_mensual    3.42
review             1.47
dtype: float64


Ninguna columna pasa del 3.5% de faltantes, así que no elimino columnas ni filas: borrar una fila por un dato faltante me haría perder las otras 20 columnas que sí están completas.

- **`ciudad` (24, 1.3%): imputo "Desconocida".** No hay forma de deducirla con otras columnas y rellenar con la moda (CDMX) sería inventar que viven ahí.
- **`ingreso_mensual` (63, 3.42%) y `minutos_app` (37, 2.01%): conservo como NaN.** El ingreso tiene una cola muy larga (máximo 326,069 contra una mediana de 30,176), así que una mediana global lo taparía. Si más adelante hay que modelar, los imputaré ahí, sin mezclar datos de prueba.
- **`satisfaccion` (46, 2.5%): conservo como NaN.** Es una opinión del cliente; rellenarla sería inventarla. En el Challenge 6 simplemente se excluyen quienes no contestaron.
- **`review` (27, 1.47%): conservo.** Que falte no es un error, ese cliente no escribió reseña.

## Paso 3 — Duplicados

In [5]:
dup = df[df.duplicated(keep=False)].sort_values("id_cliente")
print("Filas involucradas:", len(dup), "| duplicados exactos:", df.duplicated().sum(), "| ids repetidos:", df["id_cliente"].duplicated().sum())
print(dup[["id_cliente", "fecha", "edad", "ciudad", "plan", "gasto_mensual", "churn"]].head(6))

df = df.drop_duplicates(keep="first").reset_index(drop=True)
print("Filas después de quitar duplicados:", len(df))

Filas involucradas: 80 | duplicados exactos: 40 | ids repetidos: 40
     id_cliente       fecha  edad     ciudad     plan  gasto_mensual  churn
629      C00034  2026-08-21    25       CDMX   Básico         242.88      1
1473     C00034  2026-08-21    25       CDMX   Básico         242.88      1
1226     C00092  2026-06-07    50     Puebla   Básico         198.37      0
83       C00092  2026-06-07    50     Puebla   Básico         198.37      0
733      C00135  2026-03-16    51  Monterrey  Premium         922.31      0
243      C00135  2026-03-16    51  Monterrey  Premium         922.31      0
Filas después de quitar duplicados: 1800


Son **duplicados reales**. Hay 40 duplicados exactos y también 40 ids repetidos, es decir, cada id repetido es una copia idéntica en todas las columnas (por ejemplo C00034 aparece dos veces con la misma fecha, el mismo gasto de $242.88 y el mismo churn). Un registro legítimamente repetido, como un cliente con dos compras, tendría al menos otra fecha o monto. Como el dataset debe tener una fila por cliente y contarlos doble sesgaría los promedios y la tasa de churn, los eliminé conservando la primera aparición: quedaron 1,800 filas.

## Paso 4 — Texto

In [6]:
for c in ["ciudad", "plan", "canal_adquisicion", "sentimiento"]:
    df[c] = df[c].str.strip()
    print(c, "| únicos:", df[c].nunique(), "-> tras pasar a minúsculas:", df[c].str.lower().nunique(), "|", sorted(df[c].unique()))

ciudad | únicos: 6 -> tras pasar a minúsculas: 6 | ['CDMX', 'Desconocida', 'Guadalajara', 'Monterrey', 'Puebla', 'Querétaro']
plan | únicos: 3 -> tras pasar a minúsculas: 3 | ['Básico', 'Estándar', 'Premium']
canal_adquisicion | únicos: 5 -> tras pasar a minúsculas: 5 | ['Anuncio', 'Email', 'Evento', 'Orgánico', 'Referido']
sentimiento | únicos: 3 -> tras pasar a minúsculas: 3 | ['negativo', 'neutral', 'positivo']


En este dataset no hay problemas de texto. El número de valores únicos es el mismo con y sin minúsculas (6, 3, 5 y 3) y coincide con el `describe` del Paso 1 (en `ciudad` son 6 y no 5 por la categoría "Desconocida"), así que no hay variantes tipo "CDMX / cdmx / Ciudad de México". Dejé el `strip()` por seguridad y no usé `.title()` porque convertiría "CDMX" en "Cdmx".

## Paso 5 — Tipos

In [7]:
df["fecha"] = pd.to_datetime(df["fecha"])
print("Rango de fechas:", df["fecha"].min().date(), "->", df["fecha"].max().date())
df.dtypes

Rango de fechas: 2025-01-01 -> 2026-09-02


id_cliente                      str
fecha                datetime64[us]
edad                          int64
ciudad                          str
plan                            str
canal_adquisicion               str
antiguedad_meses              int64
sesiones_mes                  int64
minutos_app                 float64
tickets_soporte               int64
satisfaccion                float64
descuento_pct                 int64
ingreso_mensual             float64
gasto_mensual               float64
compras_mes                   int64
mora_dias                     int64
calificacion                  int64
review                          str
sentimiento                     str
churn                         int64
gasto_proximo_mes           float64
dtype: object

Solo `fecha` necesitaba corregirse: venía como texto y ahora es `datetime` (va del 2025-01-01 al 2026-09-02). Las demás columnas de texto (`id_cliente`, `ciudad`, `plan`, `canal_adquisicion`, `review` y `sentimiento`) sí son texto y las numéricas ya eran enteros o decimales, así que no hay números guardados como texto.

## Paso 6 — Reglas de negocio

In [8]:
reglas = {
    "edad fuera de 18-100": ~df["edad"].between(18, 100),
    "cantidades negativas": (df[["antiguedad_meses", "sesiones_mes", "minutos_app", "mora_dias"]] < 0).any(axis=1),
    "gasto o ingreso <= 0": (df["gasto_mensual"] <= 0) | (df["ingreso_mensual"] <= 0),
    "descuento fuera de 0-100": ~df["descuento_pct"].between(0, 100),
    "satisfaccion fuera de 0-10": (df["satisfaccion"] < 0) | (df["satisfaccion"] > 10),
    "calificacion fuera de 1-5": ~df["calificacion"].between(1, 5),
    "fecha en el futuro": df["fecha"] > pd.Timestamp.today(),
    "0 sesiones pero minutos > 0": (df["sesiones_mes"] == 0) & (df["minutos_app"] > 0),
    "sentimiento negativo con calificacion 5": (df["sentimiento"] == "negativo") & (df["calificacion"] == 5),
}
for nombre, cond in reglas.items():
    print(f"{nombre:42s} {cond.sum()}")
print(df.loc[reglas["0 sesiones pero minutos > 0"], "minutos_app"].describe().round(1))

edad fuera de 18-100                       0
cantidades negativas                       0
gasto o ingreso <= 0                       0
descuento fuera de 0-100                   0
satisfaccion fuera de 0-10                 0
calificacion fuera de 1-5                  0
fecha en el futuro                         0
0 sesiones pero minutos > 0                14
sentimiento negativo con calificacion 5    59
count     14.0
mean      52.1
std       49.4
min        2.8
25%       18.6
50%       31.4
75%       81.5
max      140.5
Name: minutos_app, dtype: float64


In [9]:
# Outliers con la regla del IQR: los reviso, no los borro
for c in ["ingreso_mensual", "minutos_app", "gasto_mensual"]:
    q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    lim = q3 + 1.5 * (q3 - q1)
    print(c, "| arriba de", round(lim), ":", (df[c] > lim).sum(), "| mediana", round(df[c].median()), "| máximo", round(df[c].max()))

ingreso_mensual | arriba de 53332 : 13 | mediana 30176 | máximo 326069
minutos_app | arriba de 779 : 17 | mediana 303 | máximo 8752
gasto_mensual | arriba de 947 : 30 | mediana 416 | máximo 7416


In [10]:
# Las dos incoherencias se marcan con bandera (no se borra la fila)
df["flag_uso_inconsistente"] = reglas["0 sesiones pero minutos > 0"].astype(int)
df["flag_sentimiento_contradictorio"] = reglas["sentimiento negativo con calificacion 5"].astype(int)
print(df[["flag_uso_inconsistente", "flag_sentimiento_contradictorio"]].sum())

flag_uso_inconsistente             14
flag_sentimiento_contradictorio    59
dtype: int64


De las 9 reglas, 7 no tienen ningún caso: no hay edades fuera de 18-100, cantidades negativas, gasto o ingreso menor o igual a cero, descuentos, satisfacción o calificación fuera de escala, ni fechas futuras. Las dos que sí tienen casos:

- **0 sesiones pero minutos > 0 (14 filas).** Es una contradicción, aunque los minutos son pocos (mediana de 31, entre 2.8 y 140.5), quizá un desfase de registro. No sé cuál de los dos datos es el correcto, así que no corrijo ni borro: dejo la bandera `flag_uso_inconsistente`.
- **Sentimiento negativo con calificación 5 (59 filas).** Es raro pero posible, porque el sentimiento sale del texto y la calificación es otro instrumento. Bandera `flag_sentimiento_contradictorio`.

**Outliers:** hay 13 ingresos arriba de 53,332 (máximo 326,069, unas 11 veces la mediana), 17 minutos arriba de 779 (máximo 8,752, unas 146 horas al mes) y 30 gastos arriba de 947 (máximo 7,416). Ser outlier no lo vuelve error: son montos posibles de clientes de mucho ingreso, uso o gasto, que además suelen ser los más valiosos para el negocio, así que los conservo.

## Paso 7 — Validación final

In [11]:
print("Filas antes:", len(df_original), "| después:", len(df))
print("Columnas antes:", df_original.shape[1], "| después:", df.shape[1])
print("Duplicados antes:", df_original.duplicated().sum(), "| después:", df.duplicated().sum())
nulos = df.isna().sum()
print("Celdas vacías antes:", df_original.isna().sum().sum(), "| después:", nulos.sum())
print(nulos[nulos > 0])

Filas antes: 1840 | después: 1800
Columnas antes: 21 | después: 23
Duplicados antes: 40 | después: 0
Celdas vacías antes: 197 | después: 171
minutos_app        36
satisfaccion       45
ingreso_mensual    63
review             27
dtype: int64


In [12]:
df.to_csv("dataset_limpio.csv", index=False)
print("dataset_limpio.csv guardado:", pd.read_csv("dataset_limpio.csv").shape)

dataset_limpio.csv guardado: (1800, 23)


Después de limpiar hay 1,800 filas (antes 1,840) y ya no hay duplicados. Las celdas vacías bajaron de 197 a 171: se imputaron las 24 de `ciudad` y otras 2 se fueron con los duplicados. Las 171 que quedan (minutos 36, satisfacción 45, ingreso 63, reseña 27) son las que decidí conservar. Las columnas pasaron de 21 a 23 por las dos banderas. El archivo `dataset_limpio.csv` queda con 1,800 filas y 23 columnas, una fila por cliente.

## Preguntas de reflexión

1. **¿Cuántas filas había inicialmente?** 1,840 filas y 21 columnas.

2. **¿Cuántas quedaron?** 1,800 filas, porque eliminé 40 duplicados exactos. Tiene 23 columnas: las 21 originales más las dos banderas.

3. **¿Qué problema fue más frecuente?** Los valores faltantes: 197 celdas vacías en cinco columnas (la más afectada, `ingreso_mensual`, con 63). Después vienen los 59 casos de sentimiento negativo con calificación 5 y los 40 duplicados.

4. **¿Qué decisión fue la más difícil de justificar?** Qué hacer con las filas incoherentes (las 14 con 0 sesiones y minutos, y las 59 contradictorias), porque sin un diccionario de datos no sé cuál valor es el equivocado. Borrar la fila perdería datos válidos y "corregir" un valor sería inventarlo, así que las marqué con banderas. También fue difícil no imputar `ingreso_mensual` y `minutos_app`, porque uno tiende a dejar todo sin nulos.

5. **¿Qué información pudo perderse?** Con los duplicados casi nada, eran copias exactas. En `ciudad`, "Desconocida" junta a todos los que no la reportaron. Las banderas solo indican sospecha, no confirman error. Si hubiera eliminado los outliers, habría perdido a los clientes de mayor gasto e ingreso. Y como dejé 171 celdas vacías, quien use el dataset tendrá que decidir cómo tratarlas.